# 🧠 Notebook 3: Model Architecture & Training

## CWRU Bearing Fault Detection — 1D CNN

---

### Objectives
1. Deep-dive into the 1D-CNN architecture (VibrationCNN)
2. Visualize model structure, receptive fields, and parameter counts
3. Train the model with live metric tracking
4. Analyze training dynamics (loss curves, learning rate schedule)
5. Compare results across different split strategies
6. Benchmark inference latency for edge deployment

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import time
import yaml
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import CWRUDataLoader
from src.data.preprocessing import hybrid_split, time_based_split, file_based_split
from src.models.vibration_cnn import VibrationCNN, count_parameters
from src.training.train import BearingDataset, train_model
from src.training.evaluate import evaluate_model, ModelEvaluator

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 100})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_LABELS = CWRUDataLoader.CLASS_LABELS
FS = 12000

print(f"✅ Environment ready!")
print(f"🖥️  Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 2. Model Architecture Deep-Dive

### VibrationCNN Architecture
```
Input: [batch, 1, 2048]  (single-channel vibration signal)
  │
  ├─ Block 1: Conv1d(1→32, k=64, s=8) → BN → ReLU → Dropout(0.2) → MaxPool(4)
  │    Output: [batch, 32, 32]
  │
  ├─ Block 2: Conv1d(32→64, k=32, s=1) → BN → ReLU → Dropout(0.3) → MaxPool(4)
  │    Output: [batch, 64, 8]
  │
  ├─ Block 3: Conv1d(64→128, k=16, s=1) → BN → ReLU → Dropout(0.4)
  │    Output: [batch, 128, 8]
  │
  ├─ Global Average Pooling → [batch, 128, 1]
  │
  ├─ Flatten → [batch, 128]
  │
  ├─ Linear(128→64) → ReLU → Dropout(0.5)
  │
  └─ Linear(64→10) → [batch, 10] (class logits)
```

In [ ]:
# ============================================================
# Create model and inspect architecture
# ============================================================
model = VibrationCNN(num_classes=10, dropout_rate=0.5)

print("🏗️ Model Architecture:")
print("="*60)
print(model)

total_params, trainable_params = count_parameters(model)
print(f"\n📊 Parameter Count:")
print(f"  Total:     {total_params:>10,}")
print(f"  Trainable: {trainable_params:>10,}")
print(f"  Non-trainable: {total_params - trainable_params:>10,}")

In [ ]:
# ============================================================
# Layer-by-layer parameter breakdown
# ============================================================
layer_info = []
for name, param in model.named_parameters():
    layer_info.append({
        'Layer': name,
        'Shape': str(list(param.shape)),
        'Parameters': param.numel(),
        'Trainable': param.requires_grad
    })

layer_df = pd.DataFrame(layer_info)
layer_df['% of Total'] = (layer_df['Parameters'] / layer_df['Parameters'].sum() * 100).round(1)

print("\n📋 Parameter Breakdown by Layer:")
display(layer_df)

# Visualize parameter distribution
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(layer_df['Layer'], layer_df['Parameters'], 
               color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Number of Parameters')
ax.set_title('Parameter Distribution Across Layers', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3, axis='x')
for bar, pct in zip(bars, layer_df['% of Total']):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2, 
            f'{pct}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Feature map shape progression through the network
# ============================================================
print("\n🔄 Feature Map Shape Progression:")
print("="*60)

x = torch.randn(1, 1, 2048)
print(f"Input:               {list(x.shape)}")

# Track through each layer in features
layer_names = []
shapes = [list(x.shape)]
layer_names.append('Input')

for i, layer in enumerate(model.features):
    x = layer(x)
    name = f"{type(layer).__name__}"
    if hasattr(layer, 'in_channels'):
        name += f"({layer.in_channels}→{layer.out_channels})"
    elif hasattr(layer, 'num_features'):
        name += f"({layer.num_features})"
    print(f"  features[{i:2d}] {name:40s} → {list(x.shape)}")
    shapes.append(list(x.shape))
    layer_names.append(name)

for i, layer in enumerate(model.classifier):
    x = layer(x)
    name = f"{type(layer).__name__}"
    if hasattr(layer, 'in_features'):
        name += f"({layer.in_features}→{layer.out_features})"
    print(f"  classifier[{i}] {name:40s} → {list(x.shape)}")

print(f"\nOutput:              {list(x.shape)}")

In [ ]:
# ============================================================
# Receptive field calculation
# ============================================================
print("\n📐 Receptive Field Analysis:")
print("="*60)

# Block 1: Conv(k=64, s=8) → MaxPool(k=4, s=4)
# Block 2: Conv(k=32, s=1) → MaxPool(k=4, s=4)
# Block 3: Conv(k=16, s=1)

# Receptive field calculation
rf = 1  # Start
stride_product = 1

layers = [
    ('Conv1 (k=64, s=8)', 64, 8),
    ('MaxPool1 (k=4, s=4)', 4, 4),
    ('Conv2 (k=32, s=1)', 32, 1),
    ('MaxPool2 (k=4, s=4)', 4, 4),
    ('Conv3 (k=16, s=1)', 16, 1),
]

print(f"{'Layer':<25s} {'Kernel':<10s} {'Stride':<10s} {'Receptive Field':<15s}")
print("-" * 60)

for name, k, s in layers:
    rf = rf + (k - 1) * stride_product
    stride_product *= s
    duration_ms = rf / FS * 1000
    print(f"{name:<25s} {k:<10d} {s:<10d} {rf:<10d} ({duration_ms:.2f} ms)")

print(f"\n✅ Final receptive field: {rf} samples = {rf/FS*1000:.1f} ms")
print(f"   Each output neuron 'sees' {rf/FS*1000:.1f} ms of vibration data")
print(f"   Window size: 2048 samples = {2048/FS*1000:.1f} ms")
print(f"   Coverage: {rf/2048*100:.1f}% of input window")

## 3. Prepare Data for Training

In [ ]:
# ============================================================
# Load data and create splits
# ============================================================
loader = CWRUDataLoader(data_dir=str(PROJECT_ROOT / 'data' / 'cwru'))
signals_dict, metadata_df = loader.load_all_data()
labels_dict = loader.get_labels_dict(metadata_df)

# Load config
with open(PROJECT_ROOT / 'config' / 'train_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print(f"\n📋 Training Configuration:")
for section, values in config.items():
    if isinstance(values, dict):
        print(f"  {section}:")
        for k, v in values.items():
            if isinstance(v, dict):
                print(f"    {k}: {v}")
            else:
                print(f"    {k}: {v}")
    else:
        print(f"  {section}: {values}")

In [ ]:
# ============================================================
# Create train/test split using hybrid method
# ============================================================
X_train, y_train, X_test, y_test = hybrid_split(
    signals_dict, labels_dict,
    file_train_ratio=0.7,
    time_train_ratio=0.7,
    window_size=config['data']['window_size'],
    overlap=config['data']['overlap'],
    fs=config['data']['fs'],
    seed=config['seed']
)

# Create datasets
train_dataset = BearingDataset(X_train, y_train, augment=config['training']['augment_train'])
test_dataset = BearingDataset(X_test, y_test, augment=False)

train_loader = DataLoader(train_dataset, batch_size=config['data']['batch_size'], 
                          shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=config['data']['batch_size'], 
                         shuffle=False)

print(f"\n📊 Dataset Created:")
print(f"  Train: {len(train_dataset):,} samples, {len(train_loader)} batches")
print(f"  Test:  {len(test_dataset):,} samples, {len(test_loader)} batches")
print(f"  Batch size: {config['data']['batch_size']}")

## 4. Training Loop with Visualization

In [ ]:
# ============================================================
# Initialize model and train
# ============================================================
torch.manual_seed(config['seed'])
np.random.seed(config['seed'])

model = VibrationCNN(
    num_classes=config['model']['num_classes'],
    dropout_rate=config['model']['dropout_rate']
)

print(f"🚀 Starting training on {DEVICE}...")
print(f"   Epochs: {config['training']['epochs']}")
print(f"   Learning rate: {config['training']['learning_rate']}")
print(f"   Weight decay: {config['training']['weight_decay']}")
print(f"   Early stopping patience: {config['training']['early_stopping_patience']}")

train_losses, train_accs, test_losses, test_accs, best_test_acc = train_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    num_epochs=config['training']['epochs'],
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay'],
    device=DEVICE,
    early_stopping_patience=config['training']['early_stopping_patience']
)

print(f"\n🏆 Best Test Accuracy: {best_test_acc:.2f}%")

## 5. Training Curves Analysis

In [ ]:
# ============================================================
# Comprehensive training curves
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

epochs = range(1, len(train_losses) + 1)

# Loss curves
axes[0, 0].plot(epochs, train_losses, 'b-', linewidth=2, label='Train Loss', alpha=0.8)
axes[0, 0].plot(epochs, test_losses, 'r-', linewidth=2, label='Test Loss', alpha=0.8)
axes[0, 0].fill_between(epochs, train_losses, test_losses, alpha=0.1, color='gray')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training & Test Loss', fontweight='bold', fontsize=13)
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Accuracy curves
axes[0, 1].plot(epochs, train_accs, 'b-', linewidth=2, label='Train Accuracy', alpha=0.8)
axes[0, 1].plot(epochs, test_accs, 'r-', linewidth=2, label='Test Accuracy', alpha=0.8)
axes[0, 1].axhline(y=95, color='green', linestyle='--', linewidth=1.5, alpha=0.5, label='Target (95%)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].set_title('Training & Test Accuracy', fontweight='bold', fontsize=13)
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# Generalization gap (overfitting indicator)
gap = [tr - te for tr, te in zip(train_accs, test_accs)]
axes[1, 0].plot(epochs, gap, 'purple', linewidth=2)
axes[1, 0].fill_between(epochs, 0, gap, alpha=0.2, color='purple')
axes[1, 0].axhline(y=0, color='black', linewidth=0.5)
axes[1, 0].axhline(y=5, color='orange', linestyle='--', alpha=0.5, label='Warning (5% gap)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Train Acc - Test Acc (%)')
axes[1, 0].set_title('Generalization Gap (Overfitting Indicator)', fontweight='bold', fontsize=13)
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)

# Learning rate schedule (cosine annealing)
lr_schedule = []
optimizer = optim.AdamW(model.parameters(), lr=config['training']['learning_rate'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_losses))
for _ in range(len(train_losses)):
    lr_schedule.append(scheduler.get_last_lr()[0])
    scheduler.step()

axes[1, 1].plot(epochs, lr_schedule, 'green', linewidth=2)
axes[1, 1].fill_between(epochs, 0, lr_schedule, alpha=0.1, color='green')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule (Cosine Annealing)', fontweight='bold', fontsize=13)
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Training Dynamics Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Final stats
print(f"\n📊 Training Summary:")
print(f"  Epochs trained: {len(train_losses)}")
print(f"  Final train loss: {train_losses[-1]:.4f}")
print(f"  Final test loss: {test_losses[-1]:.4f}")
print(f"  Final train accuracy: {train_accs[-1]:.2f}%")
print(f"  Final test accuracy: {test_accs[-1]:.2f}%")
print(f"  Best test accuracy: {best_test_acc:.2f}%")
print(f"  Final generalization gap: {train_accs[-1] - test_accs[-1]:.2f}%")

## 6. Inference Latency Benchmarking

Target: < 100 ms for edge deployment

In [ ]:
# ============================================================
# Inference latency benchmarking
# ============================================================
model.eval()
model = model.to(DEVICE)

# Warmup
dummy = torch.randn(1, 1, 2048).to(DEVICE)
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy)

# Benchmark
batch_sizes = [1, 8, 16, 32, 64]
latency_results = []

for bs in batch_sizes:
    dummy = torch.randn(bs, 1, 2048).to(DEVICE)
    times = []
    
    for _ in range(100):
        start = time.perf_counter()
        with torch.no_grad():
            _ = model(dummy)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        end = time.perf_counter()
        times.append((end - start) * 1000)  # ms
    
    latency_results.append({
        'Batch Size': bs,
        'Mean (ms)': np.mean(times),
        'Std (ms)': np.std(times),
        'P50 (ms)': np.percentile(times, 50),
        'P95 (ms)': np.percentile(times, 95),
        'P99 (ms)': np.percentile(times, 99),
        'Per-sample (ms)': np.mean(times) / bs
    })

latency_df = pd.DataFrame(latency_results)
latency_df = latency_df.round(3)

print("\n⏱️  Inference Latency Benchmark:")
print(f"   Device: {DEVICE}")
display(latency_df)

# Check target
single_latency = latency_df.iloc[0]['P95 (ms)']
if single_latency < 100:
    print(f"\n✅ Single-sample P95 latency: {single_latency:.2f} ms < 100 ms target")
else:
    print(f"\n⚠️ Single-sample P95 latency: {single_latency:.2f} ms exceeds 100 ms target")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([str(bs) for bs in batch_sizes], 
       [r['Mean (ms)'] for r in latency_results],
       color='steelblue', edgecolor='black', linewidth=0.5, alpha=0.8)
ax.axhline(y=100, color='red', linestyle='--', linewidth=2, label='Target (100 ms)')
ax.set_xlabel('Batch Size')
ax.set_ylabel('Latency (ms)')
ax.set_title('Inference Latency by Batch Size', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for i, r in enumerate(latency_results):
    ax.text(i, r['Mean (ms)'] + 0.5, f"{r['Mean (ms)']:.1f}", 
            ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Model Size & Deployment Metrics

In [ ]:
# ============================================================
# Model size analysis
# ============================================================
print("📦 Model Size Analysis:")
print("="*50)

# Save model temporarily to check size
import tempfile
import os

# State dict only
tmp_path = str(PROJECT_ROOT / 'notebooks' / '_tmp_model.pth')
torch.save(model.state_dict(), tmp_path)
state_dict_size = os.path.getsize(tmp_path) / (1024 * 1024)  # MB
os.remove(tmp_path)

# Full checkpoint
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config,
}, tmp_path)
full_size = os.path.getsize(tmp_path) / (1024 * 1024)  # MB
os.remove(tmp_path)

total_params, trainable_params = count_parameters(model)
param_memory = total_params * 4 / (1024 * 1024)  # float32, MB

print(f"  Total parameters: {total_params:,}")
print(f"  Parameter memory (float32): {param_memory:.2f} MB")
print(f"  State dict size: {state_dict_size:.2f} MB")
print(f"  Full checkpoint: {full_size:.2f} MB")
print(f"\n  🎯 This model is suitable for:")
print(f"     ✅ Edge devices (< 10 MB)")
print(f"     ✅ Real-time inference (< 100 ms)")
print(f"     ✅ Mobile deployment")

## 8. Feature Map Visualization

Visualize what the CNN learns at each convolutional block.

In [ ]:
# ============================================================
# Feature map visualization
# ============================================================
model.eval()
model = model.to('cpu')

# Get a test sample
sample_x = torch.FloatTensor(X_test[0]).unsqueeze(0).unsqueeze(0)  # [1, 1, 2048]
sample_label = y_test[0]

# Hook-based feature extraction
activations = {}

def get_activation(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks
hooks = []
for i, layer in enumerate(model.features):
    hook = layer.register_forward_hook(get_activation(f'features_{i}'))
    hooks.append(hook)

# Forward pass
with torch.no_grad():
    output = model(sample_x)

# Remove hooks
for hook in hooks:
    hook.remove()

# Visualize activations after each block
block_names = [
    ('features_0', 'Conv1d (1→32, k=64)'),
    ('features_2', 'ReLU after Block 1'),
    ('features_4', 'After MaxPool1'),
    ('features_5', 'Conv1d (32→64, k=32)'),
    ('features_7', 'ReLU after Block 2'),
    ('features_9', 'After MaxPool2'),
    ('features_10', 'Conv1d (64→128, k=16)'),
    ('features_12', 'ReLU after Block 3'),
]

fig, axes = plt.subplots(4, 2, figsize=(16, 16))
axes = axes.flatten()

for i, (key, name) in enumerate(block_names):
    if key in activations:
        act = activations[key].squeeze().numpy()
        
        if act.ndim == 1:
            axes[i].plot(act, linewidth=0.8, color='steelblue')
        else:
            # Show first 8 channels
            n_show = min(8, act.shape[0])
            for ch in range(n_show):
                axes[i].plot(act[ch], linewidth=0.5, alpha=0.7)
        
        axes[i].set_title(f'{name}\nShape: {list(activations[key].shape)}', 
                          fontweight='bold', fontsize=10)
        axes[i].grid(True, alpha=0.3)
        axes[i].set_ylabel('Activation')

plt.suptitle(f'Feature Maps — Class: {CLASS_LABELS[sample_label]}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Key Architecture Decisions & Rationale

| Decision | Choice | Rationale |
|----------|--------|-----------|
| **Architecture** | 1D-CNN | Directly processes raw vibration signals; captures temporal patterns |
| **First kernel** | k=64, s=8 | Large receptive field to capture bearing fault impulse patterns |
| **Channels** | 32→64→128 | Progressive feature hierarchy from low to high-level |
| **Pooling** | Global Avg Pool | Translation-invariant; reduces overfitting |
| **Regularization** | BN + Progressive Dropout (0.2→0.5) | Strong regularization for small dataset |
| **Optimizer** | AdamW + Cosine LR | Fast convergence with smooth annealing |
| **Loss** | CrossEntropy + Label Smoothing (0.1) | Prevents overconfident predictions |
| **Gradient clipping** | Max norm 1.0 | Stabilizes training with varying signal amplitudes |